# Challenge 1 · Advanced Wave Dynamics

[Start Here](../../Start_Here.ipynb) · Next: [Flow](../02_fluid/Challenge_2_Fluid_Flow.ipynb)

Predict wave displacement $u$ from $(x,y,t)$. Begin with constant speed on a square, then vary the speed and try a circular domain. In each level, write the PDE residual and check the equation, initial displacement, initial velocity, and boundary errors separately.

### Run this Challenge

1. Set `USE_REFERENCE = False` in setup and run it. `True` runs the instructor solution, ignoring your edits. Check the printed mode.
2. Complete `student_equations` in the linked `.py` file. Save with Ctrl+S / Command+S; editing an example in this notebook does not change the program.
3. Run the level's training cell, then its result cell. Each attempt gets a fresh directory. After changing code, settings or mode, rerun both cells.
4. Set `STEPS = 2` and `DEVICE = "cpu"` for a quick execution check. Choose longer runs using the held-out errors and predictions below.

The code uses PhysicsNeMo 2.2.2: a `FullyConnected` network, a SymPy `PDE`, `PhysicsInformer`, and a PyTorch optimizer.

### Comparing attempts

Start with a correct equation, then change one training choice at a time. Keep the domain, wave speed, initial/boundary conditions, and evaluation code fixed for comparable results. Write down the change, step count, and seed beside the before/after errors.

Training uses your `student_equations`; held-out PDE checks use the provided reference equation in both modes. Lower RMSE and relative error indicate improvement on those checks. These editable local results are practice feedback, not official scores or rankings. See [Assessment and feedback](../../ETC/course_materials/ASSESSMENT.md) for metric definitions.


In [ ]:
from pathlib import Path
from uuid import uuid4
import json
import os
import subprocess
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "02_challenges" / "01_wave" / "wave_l1.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open the notebook from inside the repository.")
sys.path.insert(0, str(ROOT))
from ETC.runtime.notebook import show_results, validate_settings

LAB_DIR = ROOT / "02_challenges" / "01_wave"
USE_REFERENCE = os.environ.get("AI4SCI_REFERENCE", "0").lower() in {"1", "true", "yes"}
# USE_REFERENCE = False  # Uncomment to override the server default for student exercises.
DEVICE = os.environ.get("AI4SCI_DEVICE", "auto")
STEPS = int(os.environ.get("AI4SCI_STEPS", "200"))  # Verify runtime and accuracy on the event GPU.
SEED = 42
OUTPUT_BASE = Path(os.environ.get("AI4SCI_OUTPUT_DIR", str(LAB_DIR / "outputs"))).expanduser().resolve()
RUN_DIRS = {}  # Latest attempt per level; results are never shared between levels.
RUN_COMPLETED = {}

def show_mode():
    validate_settings(DEVICE, STEPS, USE_REFERENCE)
    print("Mode: INSTRUCTOR REFERENCE; student_* edits are bypassed." if USE_REFERENCE
          else "Mode: STUDENT; saved student_* functions will run.")

show_mode()
print({"device": DEVICE, "steps": STEPS, "output_base": str(OUTPUT_BASE)})


## Level 1 · Basic 2D wave equation

The spatial domain is $[0,\pi]^2$, the time interval is $[0,2\pi]$, and $c=1$.
$$u_{tt}-c^2(u_{xx}+u_{yy})=0$$
Both the initial displacement and initial velocity are $\sin x\sin y$. The displacement is zero on all four edges.
$$u=\sin x\sin y[\cos(\sqrt2t)+\sin(\sqrt2t)/\sqrt2]$$
Differentiate this analytic solution to verify the PDE and both initial conditions. Why would the cosine term alone give the wrong initial velocity? `exact_reference` is used for validation, not as a training target.

### Code and exercise

Open [wave_l1.py](wave_l1.py) and complete the residual dictionary in `student_equations`. Trace the two initial conditions and the edge condition through `loss_terms`, then follow `optimizer.zero_grad → backward → step` in `main`.

`PhysicsInformer` computes spatial derivatives from `coordinates`; PyTorch autograd supplies the time derivatives `u__t` and `u__t__t`. Identify which derivative contributes to each term before running.


In [ ]:
RUN_COMPLETED[1] = False
result_dir = OUTPUT_BASE / f"wave_l1-{uuid4().hex}"
RUN_DIRS[1] = result_dir
command = [sys.executable, str(LAB_DIR / "wave_l1.py"),
           "--steps", str(STEPS), "--seed", str(SEED), "--device", DEVICE,
           "--output-dir", str(result_dir)]
if USE_REFERENCE:
    command.append("--reference")
show_mode()
print("Output:", result_dir)
subprocess.run(command, cwd=LAB_DIR, check=True)
RUN_COMPLETED[1] = True


### Read the results

Compare the prediction with the analytic solution, then inspect the PDE, initial-displacement, initial-velocity, and boundary errors in `metrics.json`. `reference_over_time` checks five times from $t=0$ to $2\pi$ and reports both per-time and combined solution errors; the preview and `final_reference_error` show only the middle time. Which term needs the most improvement?

The first and last held-out rows in `loss.csv` use identical points; intermediate rows use newly sampled training minibatches. `model.pt` stores the model and configuration, and `predictions.npz` stores the prediction arrays.


In [ ]:
if not RUN_COMPLETED.get(1, False):
    raise RuntimeError("The current Level 1 training run has not completed.")
result_dir = RUN_DIRS[1]
metrics = show_results(result_dir, steps=STEPS, seed=SEED, reference=USE_REFERENCE)


## Level 2 · Variable Wave Speed

The spatial domain and time interval are the same as in Level 1. The speed is $c(x,y)=1+0.5\sin x\cos y$, and the equation is
$$u_{tt}-c(x,y)^2(u_{xx}+u_{yy})=0.$$
This exercise uses the non-divergence form above. Do not replace it with $\nabla\cdot(c^2\nabla u)$.
The initial displacement is $\sin x\sin y$, the initial velocity is zero, and the displacement is zero on all four edges. Both the wave speed and the initial velocity changed from Level 1. No closed-form reference is supplied; check the PDE, initial-condition, and boundary residuals separately.

### Code and exercise

Complete `student_equations` in [wave_l2.py](wave_l2.py). Explain why derivatives of $c$ do not appear in this equation, and check the change to initial velocity in `loss_terms`. Before running, predict where propagation should be faster and slower.


In [ ]:
RUN_COMPLETED[2] = False
result_dir = OUTPUT_BASE / f"wave_l2-{uuid4().hex}"
RUN_DIRS[2] = result_dir
command = [sys.executable, str(LAB_DIR / "wave_l2.py"),
           "--steps", str(STEPS), "--seed", str(SEED), "--device", DEVICE,
           "--output-dir", str(result_dir)]
if USE_REFERENCE:
    command.append("--reference")
show_mode()
print("Output:", result_dir)
subprocess.run(command, cwd=LAB_DIR, check=True)
RUN_COMPLETED[2] = True


### Compare with Level 1

Compare the spatial pattern and fixed-point errors with Level 1. Remember that the initial velocity changed as well as the speed. To isolate the speed's effect, run a separate experiment with the same initial velocity. Small sampled residuals alone do not establish solution accuracy. An empty `reference_over_time` means no analytic reference is available, not zero error.


In [ ]:
if not RUN_COMPLETED.get(2, False):
    raise RuntimeError("The current Level 2 training run has not completed.")
result_dir = RUN_DIRS[2]
metrics = show_results(result_dir, steps=STEPS, seed=SEED, reference=USE_REFERENCE)


## Level 3 · Complex Boundaries and Circular Domain

The spatial domain is the interior of a circle of radius 1, the time interval is $[0,3]$, and $c=1$.
$$u_{tt}-\Delta u=0,\qquad u+0.5\partial_nu=0\quad\text{on }r=1.$$
$$u(x,y,0)=(1-x^2-y^2)^2\left[e^{-20((x-.3)^2+y^2)}+e^{-20((x+.3)^2+y^2)}\right],\quad u_t(x,y,0)=0.$$
The two pulses describe an initial displacement, not a continuing source. The envelope $(1-r^2)^2$ makes both the displacement and its normal derivative zero at $r=1$, so the initial field satisfies the Robin condition. 

The outward unit normal on the circle is $(x,y)$, so `boundary_residual` is $u+0.5(xu_x+yu_y)$. Robin constrains a combination of value and normal derivative during evolution; it does not require each to remain zero. It is not a perfectly absorbing boundary. No closed-form reference is supplied.

### Code and exercise

Complete `student_equations` in [wave_l3.py](wave_l3.py). The constant-speed PDE is the same as Level 1; the domain and conditions changed. Differentiate the envelope to check the initial Robin condition, then follow `boundary_residual` into `loss_terms`. Explain why replacing Robin with $u=0$ would solve a different problem.


In [ ]:
RUN_COMPLETED[3] = False
result_dir = OUTPUT_BASE / f"wave_l3-{uuid4().hex}"
RUN_DIRS[3] = result_dir
command = [sys.executable, str(LAB_DIR / "wave_l3.py"),
           "--steps", str(STEPS), "--seed", str(SEED), "--device", DEVICE,
           "--output-dir", str(result_dir)]
if USE_REFERENCE:
    command.append("--reference")
show_mode()
print("Output:", result_dir)
subprocess.run(command, cwd=LAB_DIR, check=True)
RUN_COMPLETED[3] = True


### Inspect the circular boundary

Read the Robin, PDE, and initial-condition errors separately. The initial field satisfies the Robin condition; the network must learn both conditions during training. Change one training setting at a time when comparing runs. The preview shows the middle of the time interval. Evaluate the model at other times to inspect the full evolution.


In [ ]:
if not RUN_COMPLETED.get(3, False):
    raise RuntimeError("The current Level 3 training run has not completed.")
result_dir = RUN_DIRS[3]
metrics = show_results(result_dir, steps=STEPS, seed=SEED, reference=USE_REFERENCE)


## Check your understanding

- Why are both displacement and velocity needed at the initial time?
- What changes in the PDE for variable speed, and what changes only in the boundary loss for the circle?
- Change either the learning rate or `STEPS`. Keep the seed, sample counts, PDE, initial/boundary conditions, and evaluation code fixed, then compare the before/after held-out errors.
- As a separate exploration, change a sample count or a condition weight and predict its effect. These changes can also change the evaluation points or reported loss scale in this implementation, so their held-out values are not directly comparable with the previous run.

Next: learn velocity and pressure in [Challenge 2 · Flow](../02_fluid/Challenge_2_Fluid_Flow.ipynb). [Start Here](../../Start_Here.ipynb).

Adapted from the OpenHackathons materials. [License](../../LICENSE).


--- 

Further resources: [Open Hackathons Resources](https://www.openhackathons.org/s/technical-resources). Community support: [OpenACC and Hackathons Slack Channel](https://www.openacc.org/community#slack).

---

# Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials may include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.